You potentially need to `pip install pandas` first.

Just run the cell below. Click on the output, not on the code part of the cell; that opens the whole code. If you accidentally open it, go `View -> Collapse Selected Code` to close it again.

In [105]:
group = 'rigi'

import re
from getpass import getuser
from collections import Counter, defaultdict
from itertools import chain, repeat
from pathlib import Path
from multiprocessing import Pool

import pandas as pd
import rich
from rich.table import Table
from rich.text import Text


# Workaround to fix weird scroll-inducing whitespace at end of many cell's outputs.
from IPython.display import display_html, display
display_html("""<style>.jp-OutputArea { max-height: none !important; overflow-y: visible !important }</style>""", raw=True)


########################
# READ ALL ACTIVE JOBS #


# We need to go with length and cut by lenght, because some entries may have spaces, some may be too long, etc.
jobs = !squeue -A {group} -O JobId:20,Name:20,UserName:20,State:20,TimeUsed:20,NumCPUs:20,QOS:20,NumNodes:20,GRES:20,RestartCnt:20,Reason:20
jobs = [[j[i*20:(i+1)*20].strip() for i in range(11)] for j in jobs]
jobs = pd.DataFrame(jobs[1:], columns=jobs[0])



##########################
# READ ALL WORKDIRS EVER #

# _xid_re = re.compile(r'(\d\d)(\d\d)_(\d\d)(\d\d)(\d\d)')
_xid_re = re.compile(r'\d\d\d\d_\d\d\d\d\d\d')
def extract_xid(name):
    if firstmatch := _xid_re.search(name):
        return firstmatch.group()
    return None


basedir = Path('/checkpoint/rigi/bv2/workdirs')
workdirs = [d.name for d in basedir.iterdir() if d.is_dir()]
wd_by_xid = {xid: wd for wd in workdirs if (xid := extract_xid(wd))}

#####################################
# SPLIT INTO CURRENT / RECENT / OLD #
# We do this split to add much more info to recent, and less to old.

NUM_RECENT = 50

hot_xids = {}
cold_xids = {}
for xid, wd in sorted(wd_by_xid.items(), reverse=True):
    states = Counter(jobs[jobs.NAME == xid].STATE)
    if states:
        hot_xids[xid] = {"states": states, "wd": wd}
    else:
        cold_xids[xid] = wd

frozen_xids = {xid: {"wd": cold_xids[xid]} for xid in list(cold_xids)[NUM_RECENT:]}
cold_xids = {xid: {"wd": cold_xids[xid]} for xid in list(cold_xids)[:NUM_RECENT]}

def _extra_info(xid, info):
    info["user"] = (basedir / info["wd"]).owner()
    launchinfo = basedir / info["wd"] / 'launchinfo.txt'
    if launchinfo.is_file():  # Launched with our sweep launcher
        info["wus"] = {}
        for wuwd in (basedir / info["wd"]).iterdir():
            if wuwd.is_dir():
                info["wus"][wuwd] = (wuwd / "DONE").exists()
        info["config"] = next(re.finditer(r"bv2/config/(.*?) ", launchinfo.read_text())).group(1)
    return xid, info

with Pool() as pool:
    hot_xids = dict(pool.starmap(_extra_info, hot_xids.items()))
    cold_xids = dict(pool.starmap(_extra_info, cold_xids.items()))


#########################
# PREPARE VISUALIZATION #

STATE_NAMES = {
    "RUNNING": Text("Run", "blue"),
    "PENDING": Text("Pend", "yellow"),
    "REQUEUE_HOLD": Text("Hold", "red"),
    "COMPLETING": Text("End", "green"),
    # And our own for old jobs:
    True: Text("Done", "green"),
    False: Text("Fail", "red"),
}
# STATE_NAMES = {"RUNNING": "🏃", "PENDING": "⏳", "REQUEUE_HOLD": "🚫"}  # Sadly misaligns columns.

nGPUs = {f'gres/gpu:{i}': i for i in range(1, 9)}

tblH = Table(show_header=True, header_style="bold magenta", show_footer=True, footer_style="bold magenta", box=rich.box.HORIZONTALS, collapse_padding=True)
tblH.add_column("xid", justify="left")
tblH.add_column("usr", justify="left")
tblH.add_column("states", justify="left")
tblH.add_column("wus", justify="right")
# tblH.add_column("N", justify="right")
tblH.add_column("gpu", justify="right")
tblH.add_column("gpus", justify="right")
tblH.add_column("r", justify="right")
tblH.add_column("QoS", justify="left")
tblH.add_column("config", justify="left")

all_states = Counter()
all_total_wus = 0
all_total_gpus = 0
for xid, info in hot_xids.items():
    xjobs = jobs[jobs.NAME == xid]
    qos = ' '.join(xjobs.QOS.unique().tolist())
    users = ' '.join(xjobs.USER.unique().tolist())
    nodes = ' '.join(xjobs.NODES.unique().tolist())
    gpus = int(nodes) * nGPUs.get(xjobs.TRES_PER_NODE.unique().tolist()[0], 0)
    total_gpus = gpus * info["states"]["RUNNING"]
    max_restarts = str(xjobs.RESTART_COUNT.max())

    all_total_wus += len(info["wus"])
    all_total_gpus += total_gpus

    ndone = sum(info['wus'].values())
    states = info["states"] + Counter({True: ndone, False: len(info['wus']) - ndone - sum(info["states"].values())})
    all_states.update(states)
    states = Text(' ').join(STATE_NAMES[s] + Text(f":{n}") for s, n in states.most_common())
    tblH.add_row(xid, users, states, str(len(info["wus"])), str(gpus), str(total_gpus), max_restarts, qos, info["config"])

tblH.columns[2].footer = Text(' ').join(STATE_NAMES[s] + Text(f":{n}") for s, n in all_states.most_common())
tblH.columns[3].footer = str(all_total_wus)
tblH.columns[5].footer = str(all_total_gpus)

tblC = Table(show_header=True, header_style="bold magenta", box=rich.box.HORIZONTALS)
tblC.add_column("xid", justify="left")
tblC.add_column("usr", justify="left")
tblC.add_column("wus", justify="right")
for xid, info in cold_xids.items():
    if info["user"] != getuser():
        continue
    states = []
    if nfail := sum(1 for v in info["wus"].values() if v is False):
        states.append(Text(f"{nfail}", "red"))
    if ngood := sum(1 for v in info["wus"].values() if v is True):
        states.append(Text(f"{ngood}", "green"))
    tblC.add_row(xid, info["user"], Text("+").join(states))

rich.print('All currently active experiments:', tblH,
           f'YOUR ({getuser()}) experiments in the most recent {NUM_RECENT} inactive ones:', tblC,
           f'Very old experiments: [{len(frozen_xids)} not shown]')

All currently active experiments:
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  xid          usr   states                        wus  gpu  gpus  r  QoS                   config                 
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  1129_221439  qkv   Run:22 Pend:3                  22    8   176  0  h100_rigi_high        finevision.py          
  1128_142436  zhai  Run:1                           1   64    64  0  h100_rigi_high        finevision_8n_full.py  
  1128_101645  zhai  Run:14                         14    8   112  3  h200_lowest           finevision_reg.py      
  1125_145646  zhai  Run:2                           2   64   128  1  h100_rigi_high        finevision_8n.py       
  1117_163432  qkv   Done:114 Fail:4 Hold:2        120    8     0  5  h100_lowest           code.py                
  1114_221425  qkv   Done:238 Run:1 Fail:1         240    8     8  0  h100_foundations_sha  code.py                
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
                     Done:352 Run:40 Fail:5        399        488                                                  
                     Pend:3 Hold:2                                                                                 
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
YOUR (qkv) experiments in the most recent 50 inactive ones:
 ────────────────────────── 
  xid           usr    wus  
 ────────────────────────── 
  1129_220212   qkv     25  
  1129_200121   qkv     14  
  1129_194435   qkv   11+3  
  1129_192511   qkv      6  
  1129_145744   qkv     24  
  1129_095956   qkv     24  
  1129_095248   qkv         
  1128_153823   qkv         
  1126_154737   qkv      1  
 ────────────────────────── 
Very old experiments: [238 not shown]

# Closer look at single XID (TODO)

In [ ]:
XID = ""

# Quick look at config and logs

### Code setup

In [37]:
from contextlib import contextmanager
from ipywidgets import Output
from IPython.display import display, display_html

# Workaround to fix weird scroll-inducing whitespace at end of many cell's outputs.
display_html("""<style>.jp-OutputArea { max-height: none !important; overflow-y: visible !important }</style>""", raw=True)

@contextmanager
def show_scrolling(height=200):
    out = Output(layout={"border": "1px solid #ccc", "height": f"{height}px", "overflow": "auto"})
    with out:
        yield
    display(out)

from pathlib import Path
import json
import sws

def print_config(run, height=384):
    workdir = Path('/checkpoint/rigi/bv2/workdirs') / run
    c = sws.Config(**json.loads((workdir / 'config.json').read_text())).finalize()
    with show_scrolling(height):
        print(c)


def print_logs(run, height=256, head=1000, tail=1000, linehead=100, linetail=100):
    def _snip_long_line(line, linehead=linehead, linetail=linetail):
        if len(line) > linehead + linetail:
            return line[:linehead] + " ... <SNIP> ... " + line[-linetail:]
        else:
            return line

    workdir = Path('/checkpoint/rigi/bv2/workdirs') / run
    c = sws.Config(**json.loads((workdir / 'config.json').read_text())).finalize()

    user = workdir.owner()
    full_log = Path(f'/checkpoint/rigi/bv2/slurm_out/{user}/{c.jid}.txt').read_text()
    loglines = full_log.split('\n')
    print(f"First {head} lines:")
    with show_scrolling(height):
        print('\n'.join(map(_snip_long_line, loglines[:head])))
    # import time
    # time.sleep(3)
    print(f"Last {tail} lines:")
    with show_scrolling(height):
        print('\n'.join(map(_snip_long_line, loglines[-tail:])))
    return full_log

### Actual look

In [81]:
!ls /checkpoint/rigi/bv2/workdirs/1129_220212

fv-resize-1129_220212-0/   fv-resize-1129_220212-17/  fv-resize-1129_220212-3/
fv-resize-1129_220212-1/   fv-resize-1129_220212-18/  fv-resize-1129_220212-4/
fv-resize-1129_220212-10/  fv-resize-1129_220212-19/  fv-resize-1129_220212-5/
fv-resize-1129_220212-11/  fv-resize-1129_220212-2/   fv-resize-1129_220212-6/
fv-resize-1129_220212-12/  fv-resize-1129_220212-20/  fv-resize-1129_220212-7/
fv-resize-1129_220212-13/  fv-resize-1129_220212-21/  fv-resize-1129_220212-8/
fv-resize-1129_220212-14/  fv-resize-1129_220212-22/  fv-resize-1129_220212-9/
fv-resize-1129_220212-15/  fv-resize-1129_220212-23/  launchinfo.txt
fv-resize-1129_220212-16/  fv-resize-1129_220212-24/


In [82]:
print_config('1129_220212/fv-resize-1129_220212-0', height=384)

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

In [83]:
# log = print_logs('1129_145744/fv-resize-1129_145744-14', height=192, head=1500, tail=10_000, linehead=200, linetail=200)
log = print_logs('1129_220212/fv-resize-1129_220212-0', height=192, head=1500, tail=10_000, linehead=200, linetail=200)

First 1500 lines:


Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

Last 10000 lines:


Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

# Tmp/dev

This is a place to dig deeper or figure out some things. The useful variables are: `jobs` (from `squeue` command), `hot_xids`, and `workdirs` (or sth like `basedir / workdirs[0]`).

In [31]:
jobs.query('NAME == "1125_145646"')

,JOBID,NAME,USER,STATE,TIME,CPUS,QOS,NODES,TRES_PER_NODE,RESTART_COUNT,REASON
1,1113906,1125_145646,zhai,RUNNING,2-05:52:47,1536,h100_rigi_high,8,gres/gpu:8,1,None
2,1113905,1125_145646,zhai,RUNNING,2-05:53:50,1536,h100_rigi_high,8,gres/gpu:8,1,None


In [94]:
jobs.query('NAME == "1129_221439"')

,JOBID,NAME,USER,STATE,TIME,CPUS,QOS,NODES,TRES_PER_NODE,RESTART_COUNT,REASON
0,1247521,1129_221439,qkv,PENDING,0:00,192,h100_rigi_high,1,gres/gpu:8,0,QOSGrpCpuLimit
1,1247520,1129_221439,qkv,PENDING,0:00,192,h100_rigi_high,1,gres/gpu:8,0,QOSGrpCpuLimit
2,1247519,1129_221439,qkv,PENDING,0:00,192,h100_rigi_high,1,gres/gpu:8,0,QOSGrpCpuLimit
6,1247518,1129_221439,qkv,RUNNING,10:13:06,192,h100_rigi_high,1,gres/gpu:8,0,None
7,1247517,1129_221439,qkv,RUNNING,10:33:51,192,h100_rigi_high,1,gres/gpu:8,0,None
8,1247516,1129_221439,qkv,RUNNING,10:39:22,192,h100_rigi_high,1,gres/gpu:8,0,None
9,1247515,1129_221439,qkv,RUNNING,10:42:57,192,h100_rigi_high,1,gres/gpu:8,0,None
10,1247514,1129_221439,qkv,RUNNING,10:46:12,192,h100_rigi_high,1,gres/gpu:8,0,None
11,1247513,1129_221439,qkv,RUNNING,10:47:12,192,h100_rigi_high,1,gres/gpu:8,0,None
12,1247512,1129_221439,qkv,RUNNING,10:48:44,192,h100_rigi_high,1,gres/gpu:8,0,None
